MCS 320 quiz 4 Friday 18 September 2026

# Question 1

Make a fast callable object of
$\displaystyle \frac{\cos(x) + x^2 + y}{x + y^2}$
and draw the expression tree. 

## answer to question 1

In [1]:
x, y = var('x, y')
e = (cos(x) + x^2 + y)/(x + y^2)
show(e)

(x^2 + y + cos(x))/(y^2 + x)

In [2]:
from sage.ext.fast_callable import ExpressionTreeBuilder
etb = ExpressionTreeBuilder(vars=['x','y'])

In [3]:
f = etb(e)
f

div(add(add(ipow(v_0, 2), v_1), cos(v_0)), add(ipow(v_1, 2), v_0))

In [4]:
L0 = LabelledBinaryTree([None, None], label='v_0')
L1 = LabelledBinaryTree([None, None], label='v_1')
L2 = LabelledBinaryTree([None, None], label='2')
ny2 = LabelledBinaryTree([L1, L2], label='ipow')
ascii_art(ny2)

  _ipow_
 /      \
v_1      2

In [5]:
denominator = LabelledBinaryTree([ny2, L0], label='add')
ascii_art(denominator)

     ___add___
    /         \
  _ipow_       v_0
 /      \      
v_1      2     

In [6]:
nx2 = LabelledBinaryTree([L0, L2], label='ipow')
nx2a = LabelledBinaryTree([nx2, L1], label='add')
ascii_art(nx2a)

     ___add___
    /         \
  _ipow_       v_1
 /      \      
v_0      2     

In [7]:
cosx = LabelledBinaryTree([L0,None], label='cos')
numerator = LabelledBinaryTree([nx2a, cosx], label='add')
ascii_art(numerator)

          _____add______
         /              \
     ___add___           cos
    /         \         /
  _ipow_       v_1     v_0
 /      \              
v_0      2             

In [8]:
tree = LabelledBinaryTree([numerator, denominator], label='div')
ascii_art(tree)

                 __________div__________
                /                       \
          _____add______              ___add___
         /              \            /         \
     ___add___           cos       _ipow_       v_0
    /         \         /         /      \      
  _ipow_       v_1     v_0       v_1      2     
 /      \                        
v_0      2                       

# Question 2

Consider the function 
$\displaystyle f(n) = \frac{1}{n} \sum_{k=1}^{n-1}
\sin\left( \pi \frac{k}{n} \right)$.

1. Use the above definition of $f$ to make a function ``F``
   which for some positive integer $n$ returns the floating-point 
   approximation of $f(n)$ in double precision.

2. Time the execution of ``F`` for $n=10,000$.
   Explain why ``F`` is inefficient.

3. Define the function ``V``, vectorizing ``F`` with numpy.

   Verify its correctness.

   Time the execution of ``V`` for $n=10,000$
   and compare with timings for ``F``.

## answer to question 2

In [9]:
F = lambda n: float(sum([sin(pi*k/n) for k in range(1, n)])/n)

In [10]:
timeit('F(10000)')

5 loops, best of 3: 616 ms per loop

The function F is inefficient because all 10,000 calls to the sin function with rational arguments are stored. The conversion to float happens only at the very end.

In [11]:
def V(n):
    from numpy import sin, linspace, pi, sum
    x = linspace(0, 1, n+1)
    return sum(sin(pi*x[1:n]))/n

To verify the correctness, we compute the difference between ``F(100)`` and ``V(100)``.

In [12]:
F(100) - V(100)

-2.220446049250313e-16

We observe that the difference if of the same order as the double precision.

In [13]:
timeit('V(10000)')

625 loops, best of 3: 56.9 μs per loop

We observe a dramatic reduction is time: from milliseconds to microseconds.